# Step 2A — Prompt-Only Inference (No Fine-Tuning)

Generates replies for **all users** using LLaMA 3 8B with no fine-tuning.

For each user it:
- Pulls that user’s rows from `test.csv` as the **triggers** (incoming messages)
- Samples a few of that user’s rows from `train.csv` as **few-shot style examples**
- Builds a system prompt combining the persona + examples
- Generates a reply to each trigger
- Saves results (with ground-truth replies) to `prompt_results.csv`

**Before running:** complete `1_preprocess.ipynb` first.

> ⚠️ Set runtime to **GPU**: Runtime → Change runtime type → T4 GPU

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas torch huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00


## 3. HuggingFace Login

LLaMA 3 is a gated model. Before running this:
1. Create a free account at [huggingface.co](https://huggingface.co)
2. Request access at [meta-llama/Meta-Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct) (approved in minutes)
3. Create a token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
4. Paste it when prompted below

In [ ]:
from huggingface_hub import login
login()

## 4. Configuration

Edit the paths and settings below before running anything else.

In [ ]:
# Paths to the files produced by preprocessing-script.ipynb
TRAIN_PATH  = 'train.csv'
TEST_PATH   = 'test.csv'

# Where to save the results
OUTPUT_PATH = 'few-shot-prompt-results.csv'

# Model
MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'

# Load in 4-bit quantisation to fit on Colab's T4 GPU (~6GB instead of ~16GB)
LOAD_IN_4BIT = True

# Number of real (incoming, reply) pairs from train.csv to show as style examples
NUM_EXAMPLES = 5

# Maximum number of new tokens to generate per reply
MAX_NEW_TOKENS = 128

SEED = 42

## 5. Imports & GPU Check

In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

GPU available : True
GPU name      : Tesla T4
VRAM          : 15.6 GB


## 6. Load Train & Test Data

In [ ]:
def load_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower()
    return df

train_df = load_csv(TRAIN_PATH)
test_df  = load_csv(TEST_PATH)

users = test_df['user_id'].unique().tolist()

print(f'Train rows : {len(train_df)}')
print(f'Test rows  : {len(test_df)}')
print(f'Users      : {users}')
print('\nTriggers per user (from test.csv):')
print(test_df['user_id'].value_counts())

Train rows : 2109
Test rows  : 523
Users      : ['U01', 'U03', 'U09', 'U11', 'U12', 'U14', 'U15']

Triggers per user (from test.csv):
user_id
U15    373
U09     43
U01     36
U12     28
U11     23
U14     13
U03      7
Name: count, dtype: int64


## 7. Helper Functions

In [ ]:
def get_few_shot_examples(train_df, user_id, n, seed=42):
    """
    Sample n (incoming, reply) pairs from the user's TRAIN rows only.
    These are used as style examples in the system prompt, never as triggers.
    """
    user_train = train_df[train_df['user_id'] == user_id]
    if user_train.empty:
        raise ValueError(f"No training data found for user '{user_id}'")
    sample = user_train.sample(min(n, len(user_train)), random_state=seed)
    return list(zip(sample['incoming'], sample['reply']))


def build_system_prompt(user_id, examples):
    """
    Build the persona + few-shot system prompt.
    examples: list of (incoming, reply) tuples from train.csv.
    """
    examples_block = '\n'.join(
        f'  Incoming: {inc}\n  Reply:    {rep}'
        for inc, rep in examples
    )
    return (
        f"You are impersonating user '{user_id}'. "
        f"Reply to incoming text messages exactly as they would, "
        f"matching their vocabulary, tone, punctuation, and message length.\n\n"
        f"Examples of how they write:\n\n{examples_block}\n\n"
        f"Reply to the next message in the same style. Output only the reply, nothing else."
    )


def generate_reply(tokenizer, model, system_prompt, incoming, max_new_tokens):
    """Generate a reply to an incoming message given a system prompt."""
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': incoming},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(model.device)

    if hasattr(encoded, 'input_ids'):
      input_ids = encoded.input_ids.to(model.device)  # BatchEncoding (newer transformers)
    else:
        input_ids = encoded.to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the input prompt)
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 8. Load Model

Downloads LLaMA 3 8B on first run (~16GB, cached after that).

> ⏱ Expect 5–10 minutes on first run, ~1 minute on subsequent runs.

In [ ]:
bnb_config = None
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    low_cpu_mem_usage=True,
    torch_dtype=torch.bfloat16 if not LOAD_IN_4BIT else None,
)
model.eval()
print('Model loaded successfully.')

Loading tokenizer...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded successfully.


## 9. Generate Replies — All Users

For each user:
- **Triggers** come from their rows in `test.csv` (the incoming messages to reply to)
- **Few-shot examples** come from their rows in `train.csv` (to show the model their style)

These two sets never overlap because of the split done in notebook 1.

In [ ]:
all_results = []

for user_id in users:
    print('=' * 55)
    print(f'  User: {user_id}')
    print('=' * 55)

    # Few-shot examples from train set
    examples = get_few_shot_examples(train_df, user_id, NUM_EXAMPLES, SEED)
    system_prompt = build_system_prompt(user_id, examples)

    # Triggers from test set
    user_test = test_df[test_df['user_id'] == user_id].reset_index(drop=True)
    print(f'  Triggers : {len(user_test)}')
    print(f'  Examples : {len(examples)}')
    print()

    for idx, row in user_test.iterrows():
        generated = generate_reply(
            tokenizer, model,
            system_prompt,
            row['incoming'],
            MAX_NEW_TOKENS
        )

        all_results.append({
            'user_id'        : user_id,
            'incoming'       : row['incoming'],
            'ground_truth'   : row['reply'],
            'generated_reply': generated,
        })

        print(f'  [{idx + 1}/{len(user_test)}]')
        print(f'    Incoming     : {row["incoming"]}')
        print(f'    Ground truth : {row["reply"]}')
        print(f'    Generated    : {generated}')
        print()

print('All users complete.')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  User: U01
  Triggers : 36
  Examples : 5

  [1/36]
    Incoming     : Holiiiii, Ne contajiaste el Holiiii
    Ground truth : Jajajaja sii el holii es contajiosos jaja
    Generated    : Jajajajaja, sí, me olvidé

  [2/36]
    Incoming     : Waaoooo pense que ibas a dormir
    Ground truth : No voy al culto jaja y de ahi a seguir estudiando
    Generated    : Jajajaja no, no, no!

  [3/36]
    Incoming     : Hoodoo sorry ok., Hola mi preciosa
    Ground truth : Hola Pa, Pa estoy con.la.[NAME, 5] en.un concierto cristiano
    Generated    : Haha jajaja, ¡hola mi amor!

  [4/36]
    Incoming     : Gracias a Dios, Todo bien?
    Ground truth : Si todo esta bien :) aun no se despierta
    Generated    : Si, todo bien.

  [5/36]
    Incoming     : Hijita  que va a comer, Posiblemente  hoy vaya al [NAME, 5] con el carro
    Ground truth : Estoy con la.[NAME, 4] ahi vemos q comemos
    Generated    : Hahahaha! ¡Eso sería una sorpresa!

  [6/36]
    Incoming     : [EMO, 15], Jejej, Ni yo, Ave

## 10. Save Results

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved {len(results_df)} rows to: {OUTPUT_PATH}')
print('\nResults per user:')
print(results_df['user_id'].value_counts())
results_df.head(10)

Saved 523 rows to: /content/drive/MyDrive/Spring 2026/APLN 552/results/prompt_results.csv

Results per user:
user_id
U15    373
U09     43
U01     36
U12     28
U11     23
U14     13
U03      7
Name: count, dtype: int64


,user_id,incoming,ground_truth,generated_reply
0,U01,"Holiiiii, Ne contajiaste el Holiiii",Jajajaja sii el holii es contajiosos jaja,"Jajajajaja, sí, me olvidé"
1,U01,Waaoooo pense que ibas a dormir,No voy al culto jaja y de ahi a seguir estudiando,"Jajajaja no, no, no!"
2,U01,"Hoodoo sorry ok., Hola mi preciosa","Hola Pa, Pa estoy con.la.[NAME, 5] en.un conci...","Haha jajaja, ¡hola mi amor!"
3,U01,"Gracias a Dios, Todo bien?",Si todo esta bien :) aun no se despierta,"Si, todo bien."
4,U01,"Hijita que va a comer, Posiblemente hoy vaya...","Estoy con la.[NAME, 4] ahi vemos q comemos",Hahahaha! ¡Eso sería una sorpresa!
5,U01,"[EMO, 15], Jejej, Ni yo, Aver llegaste preciosa",No pare hacer compra para la casa no hay leche...,"Jajajaja, Gracias princesita"
6,U01,Necesito que me expliques algo maÂ–ana,Dale.maÂ–ana te explico sabes q no mas hay de ...,"Ok, cuéntame qué necesitas que explique"
7,U01,"Dios te bendiga mucho tambien corazon, sabes e...","[NAME, 5] ya nada bueno hermanito que se.mejore","Ay, mi corazón, espero que te mejores pronto. ..."
8,U01,"Buenas niches, Noches",Que tenga tambien una buena noche.,Haha jajajaja ¿cómo estás?
9,U01,"Hola esta bien gracias, Como esta todo?",Todo bien nina esta jugando :) ya comio y ya l...,"Hola! Todo bien, gracias. Estoy bien, solo un ..."
